# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dyajaballh8/FlyRank_Intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

## 1. Two paper findings + my methodology questions

### Finding 1 — Search position and CTR

The paper observes a relationship between search position and click-through rate, with pages in better search positions generally showing higher CTR.

**My methodology question:**
Where does the outcome label come from, and how is CTR calculated across the different position groups? I would want to confirm that the comparison uses a consistent definition of CTR and that differences in impressions, page mix, or exposure do not explain the observed relationship.

I would treat this finding as observed and directional evidence rather than proof that improving search position will automatically cause CTR to increase.

### Finding 2 — Content length and performance

The paper observes that content length does not appear to be a strong standalone explanation for changes in content performance.

**My methodology question:**
Does the validation design support this conclusion, or could other factors such as search position, content type, or traffic exposure influence the relationship? I would also want to confirm that the outcome label is defined independently from the features being analyzed.

I would treat this as directional evidence rather than a universal conclusion that content length has no effect.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Code check

paper_findings = [
    "Search position and CTR",
    "Content length and performance"
]

methodology_questions = [
    "Where does the outcome label come from, and is CTR calculated consistently?",
    "Does the validation design separate content length from other factors?"
]

print("Paper findings:", len(paper_findings))
print("Methodology questions:", len(methodology_questions))

assert len(paper_findings) == 2
assert len(methodology_questions) == 2

print("Section 1 check passed.")


Paper findings: 2
Methodology questions: 2
Section 1 check passed.


## 2. My model under an honest split (before/after)

## 2. My model under an honest split

I compare a naive row-level split with a grouped split by client.

The naive split can place rows from the same client in both training and test data. Since pages from the same client can share similar search behavior, this can make the evaluation more optimistic.

The grouped split keeps each client entirely within either the training or test set. This provides a more honest evaluation of how the model may perform on previously unseen clients.

The comparison below reports the measured F1 score before and after the validation improvement.

The grouped result is treated as the more defensible estimate for this question.


In [7]:
# Section 2 — My model under an honest split

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
)
""")

DATA_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-05/data_0.parquet"
)


# --------------------------------------------------
# 2. Create dataset and target
# --------------------------------------------------

data_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_impressions >= 500
        AND gsc_avg_position > 0
        AND gsc_avg_position <= 20
        AND gsc_clicks * 100.0
            / NULLIF(gsc_impressions, 0) < 0.5
        THEN 1
        ELSE 0
    END AS target

FROM read_parquet('{DATA_PATH}')

WHERE gsc_data_available = TRUE
AND gsc_impressions > 0
AND gsc_avg_position > 0
"""

df = con.execute(data_query).fetchdf()

print("Rows:", len(df))
print("Clients:", df["client_hash_id"].nunique())
print("\nTarget distribution:")
print(df["target"].value_counts())


# --------------------------------------------------
# 3. Features
# --------------------------------------------------

FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]


# ==================================================
# BEFORE: Naive row-level split
# ==================================================

X = df[FEATURES]
y = df["target"]

X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_before = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model_before.fit(
    X_train_before,
    y_train_before
)

pred_before = model_before.predict(X_test_before)

before_f1 = f1_score(
    y_test_before,
    pred_before,
    zero_division=0
)


# ==================================================
# AFTER: Grouped split by client
# ==================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        groups=df["client_hash_id"]
    )
)

train_df_after = df.iloc[train_idx].copy()
test_df_after = df.iloc[test_idx].copy()


model_after = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model_after.fit(
    train_df_after[FEATURES],
    train_df_after["target"]
)

pred_after = model_after.predict(
    test_df_after[FEATURES]
)

after_f1 = f1_score(
    test_df_after["target"],
    pred_after,
    zero_division=0
)


# ==================================================
# BEFORE / AFTER COMPARISON
# ==================================================

comparison = pd.DataFrame({
    "Validation": [
        "Naive row-level split",
        "Grouped client split"
    ],
    "F1 Score": [
        before_f1,
        after_f1
    ]
})

display(comparison.round(4))

print(f"\nBefore F1: {before_f1:.4f}")
print(f"After F1:  {after_f1:.4f}")
print(f"Difference: {after_f1 - before_f1:.4f}")


# --------------------------------------------------
# Check grouped split
# --------------------------------------------------

shared_clients = (
    set(train_df_after["client_hash_id"])
    &
    set(test_df_after["client_hash_id"])
)

print("Shared clients:", len(shared_clients))

assert len(shared_clients) == 0

print("\nSection 2 check passed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 4237020
Clients: 55

Target distribution:
target
0    4169358
1      67662
Name: count, dtype: int64


,Validation,F1 Score
0,Naive row-level split,0.7224
1,Grouped client split,0.6584



Before F1: 0.7224
After F1:  0.6584
Difference: -0.0640
Shared clients: 0

Section 2 check passed.


## 3. Leakage audit

## 3. Leakage audit

I audited the final feature set against the target definition.

The target is constructed using GSC impressions, GSC average position, and CTR. CTR is calculated from GSC clicks and GSC impressions.

The model also uses GSC impressions, GSC clicks, and GSC average position as input features.

Therefore, the target is directly constructed from the same observable signals used by the model. This creates a circular dependency between the features and the target and limits how strongly the measured model performance can be interpreted.

The Week-4 rule baseline is especially affected because its thresholds are the same conditions used to define the target. Its measured F1 score of 1.0000 should not be interpreted as evidence of a perfect real-world classifier.

A stronger future design would define the target using a separate future outcome window and use only information available before the prediction cutoff as model features.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Leakage audit

target_source_features = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
}

model_features = set(FEATURES)

overlap = target_source_features.intersection(model_features)

leakage_audit = pd.DataFrame({
    "Feature": FEATURES,
    "Used in target definition": [
        feature in target_source_features
        for feature in FEATURES
    ],
    "Used as model feature": [
        feature in model_features
        for feature in FEATURES
    ]
})

display(leakage_audit)

print("Features used in both target construction and model:")

for feature in sorted(overlap):
    print("-", feature)

print("\nNumber of overlapping features:", len(overlap))

assert len(overlap) > 0

print("\nSection 3 check passed.")

,Feature,Used in target definition,Used as model feature
0,gsc_impressions,True,True
1,gsc_clicks,True,True
2,gsc_avg_position,True,True


Features used in both target construction and model:
- gsc_avg_position
- gsc_clicks
- gsc_impressions

Number of overlapping features: 3

Section 3 check passed.


## 4. Claim rewrite

## 4. Claim rewrite

### Original claim

The Logistic Regression model accurately predicts CTR improvement opportunities.

### Safer claim

The Logistic Regression model showed measured performance in identifying content items matching the defined CTR-opportunity proxy on the evaluated client-grouped split.

The results provide directional evidence that observable GSC signals can support content-review prioritization.

However, this should be treated as decision-support rather than proof that changing a content item will improve its CTR.

The leakage audit also shows that the current target is closely connected to the model features. Stronger claims would require a future-outcome target and a leakage-resistant feature design.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Claim language check

safe_claim = """
The Logistic Regression model showed measured performance in identifying
content items matching the defined CTR-opportunity proxy on the evaluated
client-grouped split. The results provide directional evidence that observable
GSC signals can support content-review prioritization. This should be treated
as decision-support rather than proof of future CTR improvement.
"""

required_language = [
    "measured",
    "directional",
    "decision-support"
]

print("Claim:")
print(safe_claim)

print("\nClaim language check:")

for phrase in required_language:
    found = phrase in safe_claim.lower()
    print(f"{phrase}: {found}")

assert all(
    phrase in safe_claim.lower()
    for phrase in required_language
)

print("\nSection 4 check passed.")

Claim:

The Logistic Regression model showed measured performance in identifying
content items matching the defined CTR-opportunity proxy on the evaluated
client-grouped split. The results provide directional evidence that observable
GSC signals can support content-review prioritization. This should be treated
as decision-support rather than proof of future CTR improvement.


Claim language check:
measured: True
directional: True
decision-support: True

Section 4 check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.